# Phase 2: Model Training & Evaluation
## Real-World UCI Dataset (4,424 students)
Training and comparing 4 ML classifiers: Logistic Regression, Decision Tree, Random Forest, XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
TEST_SIZE = 0.3

os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)
print('Libraries loaded successfully.')

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv('../data/real_student_data.csv')
print(f'Original dataset: {df.shape}')
print(f'Target distribution:\n{df["Target"].value_counts()}')

# Binary classification: Dropout vs Graduate (remove Enrolled)
df = df[df['Target'] != 'Enrolled'].copy()
df['Dropout'] = (df['Target'] == 'Dropout').astype(int)
df.drop(columns=['Target'], inplace=True)

print(f'\nAfter filtering: {df.shape[0]} records')
print(f'Dropout: {df["Dropout"].sum()} ({df["Dropout"].mean()*100:.1f}%)')
print(f'Graduate: {(1-df["Dropout"]).sum().astype(int)} ({(1-df["Dropout"].mean())*100:.1f}%)')

In [ ]:
# --- Define Features and Target ---
feature_cols = [col for col in df.columns if col != 'Dropout']

X = df[feature_cols].copy()
y = df['Dropout'].copy()

print(f'Features: {len(feature_cols)} columns')
print(f'Target: Dropout (0=Graduate, 1=Dropout)')
print(f'\nMissing values: {X.isnull().sum().sum()}')
print(f'\nFeature list:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# --- Min-Max Normalization ---
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols, index=X.index)

print('Min-Max Scaling applied to all 36 features.')
print(f'All features now in range [0, 1]')

In [ ]:
# --- Train/Test Split (70/30 Stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')
print(f'\nTrain dropout rate: {y_train.mean()*100:.1f}%')
print(f'Test dropout rate:  {y_test.mean()*100:.1f}%')
print(f'\nClass balance preserved via stratified split.')

## 2. Train All Models
Using `scale_pos_weight` for XGBoost and `class_weight='balanced'` for others to handle imbalanced data.

In [ ]:
# Calculate class weight ratio for imbalanced handling
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count
print(f'Class imbalance ratio: {scale_weight:.2f} (Graduate/Dropout)')

models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE, max_depth=8, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        random_state=RANDOM_STATE, n_estimators=200, max_depth=12, class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        random_state=RANDOM_STATE, n_estimators=200, max_depth=6,
        learning_rate=0.1, scale_pos_weight=scale_weight,
        use_label_encoder=False, eval_metric='logloss'
    )
}

results = {}
trained_models = {}

print(f'\nTraining all models...\n')
print(f'{"Model":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10} {"ROC-AUC":>10}')
print('-' * 80)

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'ROC-AUC': auc,
        'y_pred': y_pred, 'y_proba': y_proba
    }
    
    print(f'{name:<25} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc:>10.4f}')

print('\nAll models trained successfully!')

## 3. Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k not in ['y_pred', 'y_proba']}
    for name, metrics in results.items()
}).T

comparison_df = comparison_df.round(4)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print('=== MODEL COMPARISON TABLE (sorted by ROC-AUC) ===')
comparison_df

In [ ]:
# --- Visual Comparison Bar Chart ---
fig, ax = plt.subplots(figsize=(14, 6))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.18
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=model_name, color=colors[i],
                  edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison — Real-World Dataset', fontsize=16, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for idx, (name, metrics) in enumerate(results.items()):
    cm = confusion_matrix(y_test, metrics['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Graduate', 'Dropout'],
                yticklabels=['Graduate', 'Dropout'],
                linewidths=1, linecolor='black')
    axes[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for idx, (name, metrics) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, metrics['y_proba'])
    auc_val = metrics['ROC-AUC']
    ax.plot(fpr, tpr, color=colors[idx], lw=2.5,
            label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Model Comparison (Real-World Data)', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])

plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-Validation (5-Fold)

In [ ]:
print('=== 5-Fold Cross-Validation Results ===')
print(f'{"Model":<25} {"Mean Accuracy":>15} {"Std Dev":>10}')
print('-' * 55)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    cv_results[name] = {'Mean': scores.mean(), 'Std': scores.std(), 'Scores': scores}
    print(f'{name:<25} {scores.mean():>15.4f} {scores.std():>10.4f}')

print('\nCross-validation confirms model stability across folds.')

In [ ]:
# --- Cross-Validation Box Plot ---
fig, ax = plt.subplots(figsize=(10, 6))

cv_data = [cv_results[name]['Scores'] for name in models.keys()]
bp = ax.boxplot(cv_data, labels=list(models.keys()), patch_artist=True, widths=0.5)

colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('5-Fold Cross-Validation — Accuracy Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Tree-Based Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for idx, (name, color) in enumerate([('Decision Tree', '#e67e22'), ('Random Forest', '#2ecc71'), ('XGBoost', '#e74c3c')]):
    importance = trained_models[name].feature_importances_
    sorted_idx = np.argsort(importance)
    top_n = sorted_idx[-15:]  # Top 15 features
    
    axes[idx].barh(np.array(feature_cols)[top_n], importance[top_n], color=color)
    axes[idx].set_title(f'{name} — Top 15 Features', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Feature Importance')

plt.suptitle('Feature Importance — Tree-Based Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. SHAP Explainability (XGBoost)

In [ ]:
xgb_model = trained_models['XGBoost']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values computed for {X_test.shape[0]} test samples.')
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# --- Global SHAP Feature Importance ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols,
                  plot_type='bar', show=False, max_display=15)
plt.title('Global SHAP Feature Importance — Top 15 (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SHAP Summary Plot (Beeswarm) ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False, max_display=15)
plt.title('SHAP Summary Plot — Feature Impact Direction & Magnitude', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SHAP Waterfall — High Risk vs Low Risk Student ---
high_risk_indices = y_test[y_test == 1].index
low_risk_indices = y_test[y_test == 0].index

high_risk_pos = list(X_test.index).index(high_risk_indices[0])
low_risk_pos = list(X_test.index).index(low_risk_indices[0])

fig = plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
shap.waterfall_plot(shap.Explanation(
    values=shap_values[high_risk_pos],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_pos],
    feature_names=feature_cols
), show=False, max_display=12)
plt.title('Dropout Student — SHAP Explanation', fontsize=11, fontweight='bold')

plt.subplot(1, 2, 2)
shap.waterfall_plot(shap.Explanation(
    values=shap_values[low_risk_pos],
    base_values=explainer.expected_value,
    data=X_test.iloc[low_risk_pos],
    feature_names=feature_cols
), show=False, max_display=12)
plt.title('Graduate Student — SHAP Explanation', fontsize=11, fontweight='bold')

plt.suptitle('SHAP Waterfall — Individual Student Explanations', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('../results/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Classification Reports

In [ ]:
for name, metrics in results.items():
    print(f'\n{"=" * 55}')
    print(f'  {name} — Classification Report')
    print(f'{"=" * 55}')
    print(classification_report(y_test, metrics['y_pred'],
                                target_names=['Graduate', 'Dropout']))

## 10. Save Best Model (XGBoost)

In [ ]:
# --- Save XGBoost model ---
xgb_model.save_model('../models/xgboost_model.json')
print('XGBoost model saved: ../models/xgboost_model.json')

# --- Save scaler parameters ---
scaler_params = {
    'feature_columns': feature_cols,
    'min_values': scaler.data_min_.tolist(),
    'max_values': scaler.data_max_.tolist(),
    'scale': scaler.scale_.tolist(),
    'data_range': scaler.data_range_.tolist()
}
with open('../models/scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)
print('Scaler params saved: ../models/scaler_params.json')

# --- Save comparison results ---
comparison_df.to_csv('../results/model_comparison.csv')
print('Comparison saved: ../results/model_comparison.csv')

In [ ]:
# --- Verify saved model ---
loaded_model = XGBClassifier()
loaded_model.load_model('../models/xgboost_model.json')
loaded_preds = loaded_model.predict(X_test)
match = (loaded_preds == results['XGBoost']['y_pred']).all()
print(f'Model save/load verification: {"PASSED" if match else "FAILED"}')

## 11. Final Summary

In [ ]:
best_model = comparison_df.index[0]
best_metrics = comparison_df.iloc[0]

print('=' * 65)
print('    PHASE 2 — MODEL TRAINING SUMMARY (REAL-WORLD DATA)')
print('=' * 65)

print(f'\n  Dataset:        UCI #697 — Portuguese University')
print(f'  Records:        {len(df)} students ({X_train.shape[0]} train / {X_test.shape[0]} test)')
print(f'  Features:       {len(feature_cols)} predictive features')
print(f'  Imbalance:      {df["Dropout"].mean()*100:.1f}% dropout (handled via class weights)')
print(f'  Models Trained: 4 (LogReg, DecTree, RandomForest, XGBoost)')

print(f'\n  BEST MODEL: {best_model}')
print(f'  ---------------------------------')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    print(f'  {metric:<12}: {best_metrics[metric]:.4f}')

print(f'\n  SHAP Top 5 Features (Global Importance):')
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
for i, (feat, val) in enumerate(shap_importance.head(5).items(), 1):
    print(f'    {i}. {feat}: {val:.4f}')

print(f'\n  Files Saved:')
print(f'    models/xgboost_model.json')
print(f'    models/scaler_params.json')
print(f'    results/model_comparison.csv & .png')
print(f'    results/confusion_matrices.png')
print(f'    results/roc_curves.png')
print(f'    results/cross_validation.png')
print(f'    results/feature_importance.png')
print(f'    results/shap_global_importance.png')
print(f'    results/shap_summary_plot.png')
print(f'    results/shap_waterfall.png')
print('=' * 65)